# After-visit summary lines -> billable MRF codes

This notebook takes the patient-facing **after-visit summary (AVS)** produced in `03_after_visit_summary.ipynb` and, **line by line**, maps each item to a **billable code that the hospitals actually publish** in their machine-readable files (MRFs).

Pipeline per line:

1. **Extract** — the line of clinical text is sent to the agent in `random/text_to_codes_agent.py`, which turns it into discrete billable service mentions (with status: performed / ordered / planned / historical / discussed).
2. **Map** — each codeable mention is matched against the **MRF inventory** (the real menu of `CPT` / `HCPCS` codes the hospitals price). A code outside that inventory cannot be priced, so hallucinations are dropped at validation.
3. **Price** — for each chosen code we pull the cross-hospital price range.

> Why CPT/HCPCS? Hospital MRFs bill in **CPT, HCPCS, CDM, MS-DRG, RC, CMG, LOCAL** and contain **zero SNOMED** codes. CPT/HCPCS are the line-level, cross-hospital-comparable, priceable code types, so they are the mapping target.

**Inputs:** `processed_csv/patient_to_after_visit_summary.csv`, `cms_hpt/mrfs/`, `ANTHROPIC_API_KEY` from `.env`.
**Output:** `processed_csv/avs_line_to_billable_codes.csv` — one row per (`model`, line, code). The run is **resumable** and **model-tagged**: rerunning with the same model does no work, switching models adds new rows without overwriting the old ones.

In [9]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 300)

# Resolve project paths whether the notebook runs from repo root or data_processing/
HERE = Path.cwd()
PROC = HERE if (HERE / "random").is_dir() else HERE / "data_processing"
assert (PROC / "random").is_dir(), f"Could not locate data_processing/ from {HERE}"

# The agent lives in data_processing/random and reads ANTHROPIC_API_KEY from env.
sys.path.insert(0, str(PROC / "random"))
load_dotenv(PROC.parent / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not found (check .env)"

import text_to_codes_agent as agent  # noqa: E402

# Keep the agent's map cache next to this notebook so reruns are cheap.
agent.CACHE_PATH = PROC / "text_map_cache.json"

MRF_DIR = PROC / "cms_hpt" / "mrfs"
AVS_CSV = PROC / "processed_csv" / "patient_to_after_visit_summary.csv"
OUT_CSV = PROC / "processed_csv" / "avs_line_to_billable_codes.csv"
print("MRF dir:", MRF_DIR)
print("AVS csv:", AVS_CSV)

MRF dir: /Users/robbertstruyven/Dropbox/Mac/Downloads/abridge_hackathon/abridge-hackathon/data_processing/cms_hpt/mrfs
AVS csv: /Users/robbertstruyven/Dropbox/Mac/Downloads/abridge_hackathon/abridge-hackathon/data_processing/processed_csv/patient_to_after_visit_summary.csv


## Choose the model (dumb / balanced / smart)

The agent hardcodes one model; here we override `agent.MODEL` so you can trade
cost/speed against accuracy. Start on the **cheap** model while iterating, switch
to **balanced** or **smart** for the final run. You can also set `AGENT_MODEL` in
`.env` to force a specific model id.

In [10]:
import re

# Named tiers -> concrete Anthropic model ids (see `client.models.list()`).
MODELS = {
    "dumb":     "claude-haiku-4-5-20251001",  # cheapest / fastest
    "balanced": "claude-sonnet-5",            # the agent's original default
    "smart":    "claude-opus-4-8",            # most capable / most expensive
}

# Pick a tier here, or override with a raw model id via AGENT_MODEL in .env.
MODEL_CHOICE = "dumb"  # "dumb" | "balanced" | "smart"
MODEL_CHOICE = "balanced"  # "dumb" | "balanced" | "smart"

agent.MODEL = os.environ.get("AGENT_MODEL") or MODELS[MODEL_CHOICE]

# The agent's map cache is not model-aware, so scope it per model. Otherwise a
# cheaper model's cached answers would be reused when you switch to a smarter one.
_safe = re.sub(r"[^A-Za-z0-9]+", "-", agent.MODEL)
agent.CACHE_PATH = PROC / f"text_map_cache.{_safe}.json"

print(f"Using model: {agent.MODEL}  (tier: {MODEL_CHOICE})")
print(f"Map cache:   {agent.CACHE_PATH.name}")

Using model: claude-sonnet-5  (tier: balanced)
Map cache:   text_map_cache.claude-sonnet-5.json


## Build the MRF billable-code inventory (once)

Index every `CPT`/`HCPCS` code the six hospitals publish. This is the menu the mapping stage is allowed to choose from, and the source of prices.

In [11]:
from collections import Counter
from statistics import mean, median

by_hospital = agent.load_mrf_dir(str(MRF_DIR))
inventory = agent.build_inventory(by_hospital)

print(f"\n{len(inventory):,} distinct billable codes across {len(by_hospital)} hospitals")
display(pd.Series(Counter(t for _, t in inventory)).rename("codes").to_frame())


def _mrf_label(fname):
    """Reproduce the agent's filename -> hospital label derivation."""
    lbl = re.sub(r"\.(csv|json|zip)$", "", fname, flags=re.I)
    return re.sub(r"_?standardcharges.*$", "", lbl, flags=re.I).strip("_-") or fname


# by_hospital keys are filename-derived labels (e.g. "PriceTransparency20262320").
# Map each to the real hospital name from manifest.tsv, keeping the label too.
HOSPITAL_NAMES = {}
manifest_path = MRF_DIR / "manifest.tsv"
if manifest_path.exists():
    for _, m in pd.read_csv(manifest_path, sep="\t").iterrows():
        f = m.get("local_file")
        if isinstance(f, str) and f:
            HOSPITAL_NAMES[_mrf_label(f)] = m["location_name"]
for label in by_hospital:
    HOSPITAL_NAMES.setdefault(label, label)


# ---- Per-hospital price index (built once) -----------------------------------
# For every billable (code, type) we collect, per hospital, the raw values needed
# to expose all CMS price fields: payer-negotiated dollars (mean/median/min/max/
# n_payers), gross, discounted_cash, the hospital's own min/max, and p10/p90.
_LEVEL = (("gross", "gross"), ("discounted_cash", "discounted_cash"),
          ("minimum", "min_published"), ("maximum", "max_published"),
          ("median", "p50"), ("p10", "p10"), ("p90", "p90"))


def build_price_index(by_hospital):
    idx = {}
    for hosp, recs in by_hospital.items():
        for r in recs:
            for key in r["codes"]:
                if key[1] not in agent.BILLABLE:
                    continue
                d = idx.setdefault(key, {}).setdefault(hosp, {
                    "negotiated": [], "gross": [], "discounted_cash": [],
                    "min_published": [], "max_published": [],
                    "p50": [], "p10": [], "p90": [],
                })
                if r.get("dollar") is not None:
                    d["negotiated"].append(r["dollar"])
                for src, dst in _LEVEL:
                    v = r.get(src)
                    if v is not None:
                        d[dst].append(v)
    return idx


PRICE_INDEX = build_price_index(by_hospital)


def hospital_prices(code, ctype):
    """All CMS price fields for one code, per hospital (keyed by MRF label)."""
    out = {}
    for hosp, d in PRICE_INDEX.get((str(code), str(ctype).upper()), {}).items():
        neg = sorted(d["negotiated"])
        out[hosp] = {
            "hospital_name": HOSPITAL_NAMES.get(hosp, hosp),
            # self-pay / uninsured price -- often the most actionable number
            "discounted_cash": min(d["discounted_cash"]) if d["discounted_cash"] else None,
            # what insurers actually negotiated (the real insured price)
            "negotiated_mean": round(mean(neg), 2) if neg else None,
            "negotiated_median": round(median(neg), 2) if neg else None,
            "negotiated_min": neg[0] if neg else None,
            "negotiated_max": neg[-1] if neg else None,
            "n_payers": len(neg),  # confidence: 1 payer -> don't trust the spread
            # sticker price nobody pays
            "gross": max(d["gross"]) if d["gross"] else None,
            # hospital's own de-identified min/max over ALL payers (usually wider)
            "min_published": min(d["min_published"]) if d["min_published"] else None,
            "max_published": max(d["max_published"]) if d["max_published"] else None,
            # estimated allowed amount (median_amount / estimated_amount)
            "p50": round(median(d["p50"]), 2) if d["p50"] else None,
            # percentiles of amounts actually paid (12 mo of remittance data)
            "p10": min(d["p10"]) if d["p10"] else None,
            "p90": max(d["p90"]) if d["p90"] else None,
        }
    return out


display(pd.Series(HOSPITAL_NAMES, name="hospital_name").rename_axis("mrf_label").to_frame())
print(f"Price index built for {len(PRICE_INDEX):,} billable codes.")

  04-2774441_Boston-Childrens-Longwood: 465,548 rows
  042704686_lahey-hospital-medical-center-burlington: 91,487 rows
  042769210_south-shore-hospital: 167,552 rows
  042987822_Encompass-Health-Rehabilitation-Hospital-of-Western-Massachusetts: 48,046 rows
  PriceTransparency20262320: 1,022,366 rows



20,705 distinct billable codes across 5 hospitals


,codes
CPT,4891
HCPCS,15814


,hospital_name
mrf_label,
PriceTransparency20262320,Cambridge Health Alliance
04-2774441_Boston-Childrens-Longwood,Boston Children's Longwood
042704686_lahey-hospital-medical-center-burlington,Lahey Hospital & Medical Center
042769210_south-shore-hospital,South Shore Hospital
042987822_Encompass-Health-Rehabilitation-Hospital-of-Western-Massachusetts,Encompass Health Rehabilitation Hospital of Western Massachusetts


Price index built for 20,705 billable codes.


## Explode every AVS into individual lines

Each after-visit summary is split into its bullet lines (section headers and the
`Visit summary` title dropped). Every line keeps its section (`what_we_discussed`
vs `next_steps`) so the billable next-steps can be told apart from diagnoses.

In [12]:
import re

avs_df = pd.read_csv(AVS_CSV)

HEADERS = {"visit summary", "what we discussed", "next steps",
           "your next steps", "plan", "what happens next"}
SECTION_OF = {
    "what we discussed": "what_we_discussed",
    "next steps": "next_steps", "your next steps": "next_steps",
    "plan": "next_steps", "what happens next": "next_steps",
}


def clean_line(line):
    return re.sub(r"^[\s\u2022\u2023\u25e6\-\*\u2013]+", "", line).strip()


def explode_avs(avs_text):
    """Yield (section, line_text) for every content line of an AVS."""
    section = None
    for raw in str(avs_text).splitlines():
        line = clean_line(raw)
        if not line:
            continue
        norm = line.lower().rstrip(":")
        if norm in HEADERS:
            section = SECTION_OF.get(norm)  # None for the "Visit summary" title
            continue
        yield (section or "other", line)


line_rows = []
for _, r in avs_df.iterrows():
    for line_no, (section, text) in enumerate(explode_avs(r["after_visit_summary"])):
        line_rows.append({
            "patient_number": r["patient_number"],
            "patient": r["patient"],
            "encounter_date": r["encounter_date"],
            "facility": r["facility"],
            "section": section,
            "line_number": line_no,
            "line_text": text,
        })

lines_df = pd.DataFrame(line_rows)
print(f"{len(lines_df)} AVS lines across {lines_df['patient_number'].nunique()} patients")
display(lines_df["section"].value_counts().rename("lines").to_frame())
display(lines_df.head(12))

342 AVS lines across 25 patients


,lines
section,
next_steps,198
what_we_discussed,144


,patient_number,patient,encounter_date,facility,section,line_number,line_text
0,0,Ali Jerry Kuhic,2022-08-05,WHITLEY WELLNESS LLC,what_we_discussed,0,Gingivitis
1,0,Ali Jerry Kuhic,2022-08-05,WHITLEY WELLNESS LLC,what_we_discussed,1,Chronic intractable migraine without aura
2,0,Ali Jerry Kuhic,2022-08-05,WHITLEY WELLNESS LLC,what_we_discussed,2,Social isolation and stress
3,0,Ali Jerry Kuhic,2022-08-05,WHITLEY WELLNESS LLC,what_we_discussed,3,Transportation insecurity
4,0,Ali Jerry Kuhic,2022-08-05,WHITLEY WELLNESS LLC,what_we_discussed,4,Preventive health and screening
5,0,Ali Jerry Kuhic,2022-08-05,WHITLEY WELLNESS LLC,next_steps,5,Patient referral for dental care placed (sliding-scale clinic).
6,0,Ali Jerry Kuhic,2022-08-05,WHITLEY WELLNESS LLC,next_steps,6,Daily flossing and twice-daily brushing reviewed.
7,0,Ali Jerry Kuhic,2022-08-05,WHITLEY WELLNESS LLC,next_steps,7,Medication reconciliation completed and chart list updated.
8,0,Ali Jerry Kuhic,2022-08-05,WHITLEY WELLNESS LLC,next_steps,8,Keep a simple headache log; return precautions reviewed for any change in pattern or new neurologic symptoms.
9,0,Ali Jerry Kuhic,2022-08-05,WHITLEY WELLNESS LLC,next_steps,9,Assessment of health and social care needs completed (PRAPARE).


## Code each line with the agent (extract -> map -> price)

For each line we run the two-stage agent and then price every code it chose.
Non-billable lines (a diagnosis, a lifestyle-counseling note, a declined test)
correctly yield **no code** and are kept in the output with a reason, so the
mapping stays line-complete and auditable.

**Resumable & model-aware.** Every output row is tagged with the `model` that
produced it. On each run we load the existing CSV and **skip any (model, patient,
line) already done**, so:

- rerunning with the **same model** does no API work (unless lines are missing or
  previously errored),
- switching to a **new model** processes every line again and *adds* its rows
  next to the other models' — nothing is overwritten.

Progress is flushed to the CSV every `SAVE_EVERY` lines, so an interrupted run
picks up where it left off. `MAX_LINES` lets you smoke-test first (`None` = all).

In [13]:
MAX_LINES = None  # e.g. 15 to smoke-test; None = every line
SAVE_EVERY = 10   # flush progress to the CSV every N newly-processed lines

import json

COLUMNS = [
    "model",
    "patient_number", "patient", "encounter_date", "facility",
    "section", "line_number", "line_text",
    "mention", "search_term", "category", "status", "laterality", "approach",
    "code", "code_type", "code_description",
    "confidence", "needs_review", "note", "evidence",
    # aggregates across hospitals (negotiated dollars + self-pay cash)
    "price_low", "price_high", "cash_low", "cash_high", "n_hospitals", "hospitals",
    # full per-hospital detail (JSON: cash, negotiated mean/median/min/max,
    # n_payers, gross, min/max_published, p10/p90) keyed by MRF label
    "prices_by_hospital",
]


def load_existing():
    if OUT_CSV.exists():
        # Keep codes as strings so leading zeros (e.g. anesthesia CPT 00100) survive.
        df = pd.read_csv(OUT_CSV, dtype={"code": "string", "code_type": "string"})
        if "model" not in df.columns:
            df["model"] = pd.NA
        return df
    return pd.DataFrame(columns=COLUMNS)


def price_summary(code, ctype):
    """Per-hospital CMS price fields + cross-hospital aggregates for one code."""
    per = hospital_prices(code, ctype)
    if not per:
        return None
    neg_mins = [e["negotiated_min"] for e in per.values() if e["negotiated_min"] is not None]
    neg_maxs = [e["negotiated_max"] for e in per.values() if e["negotiated_max"] is not None]
    cash = [e["discounted_cash"] for e in per.values() if e["discounted_cash"] is not None]
    return {
        "price_low": min(neg_mins) if neg_mins else None,
        "price_high": max(neg_maxs) if neg_maxs else None,
        "cash_low": min(cash) if cash else None,
        "cash_high": max(cash) if cash else None,
        "n_hospitals": len(per),
        "hospitals": "; ".join(sorted(per)),
        "prices_by_hospital": json.dumps(per),
    }


def code_line(row, model):
    """Run the agent on one AVS line; return a list of output-row dicts."""
    base = {
        "model": model,
        "patient_number": int(row.patient_number), "patient": row.patient,
        "encounter_date": row.encounter_date, "facility": row.facility,
        "section": row.section, "line_number": int(row.line_number),
        "line_text": row.line_text,
    }
    services = agent.map_services(agent.extract_services(row.line_text) or [], inventory)
    rows = []
    for s in services:
        # The extractor adds a visit-level "office visit" E/M to every line; that
        # is a per-encounter charge, not a per-line one, so drop it here.
        if s.get("category") == "evaluation_management":
            continue
        svc = {
            **base,
            "mention": s.get("mention"), "search_term": s.get("search_term"),
            "category": s.get("category"), "status": s.get("status"),
            "laterality": s.get("laterality"), "approach": s.get("approach"),
            "confidence": s.get("confidence"), "needs_review": s.get("needs_review"),
            "note": s.get("note"), "evidence": s.get("evidence"),
        }
        codes = s.get("codes") or []
        if not codes:
            rows.append(svc)  # codeable service but no priceable code, or skipped status
            continue
        for x in codes:
            code, ctype = x.get("code"), (x.get("type") or "").upper()
            desc = inventory.get((code, ctype), {}).get("desc")
            rows.append({**svc, "code": code, "code_type": ctype,
                         "code_description": desc, **(price_summary(code, ctype) or {})})
    return rows or [{**base, "status": "no_billable_service"}]


MODEL = agent.MODEL
existing = load_existing()

# Lines already done for THIS model (errored lines are not "done" -> retried).
mask_done = existing["model"].eq(MODEL)
if "status" in existing:
    mask_done &= existing["status"].ne("error")
done = {(int(p), int(l)) for p, l in
        zip(existing.loc[mask_done, "patient_number"], existing.loc[mask_done, "line_number"])}

# Keep every existing row except this model's rows for lines we are about to redo.
keep = existing[[
    not (m == MODEL and (int(p), int(l)) not in done)
    for m, p, l in zip(existing["model"], existing["patient_number"], existing["line_number"])
]] if len(existing) else existing

work = lines_df if MAX_LINES is None else lines_df.head(MAX_LINES)
todo = [r for r in work.itertuples(index=False)
        if (int(r.patient_number), int(r.line_number)) not in done]
print(f"Model {MODEL}: {len(done)} lines already done, {len(todo)} to process.")


def flush(new_rows):
    combined = pd.concat([keep, pd.DataFrame(new_rows)], ignore_index=True)
    combined.reindex(columns=COLUMNS).to_csv(OUT_CSV, index=False)
    return combined


new_rows = []
for i, r in enumerate(todo, start=1):
    try:
        new_rows.extend(code_line(r, MODEL))
    except Exception as e:  # keep going on any single-line failure
        new_rows.append({"model": MODEL, "patient_number": int(r.patient_number),
                         "patient": r.patient, "section": r.section,
                         "line_number": int(r.line_number), "line_text": r.line_text,
                         "status": "error", "note": f"{type(e).__name__}: {e}"})
        print(f"  ! line {i}: {type(e).__name__}: {e}")
    if i % SAVE_EVERY == 0 or i == len(todo):
        flush(new_rows)
        print(f"  processed {i}/{len(todo)} new lines (saved)")

codes_out_df = (flush(new_rows) if todo else keep).reindex(columns=COLUMNS)
print(f"\nDone. CSV now holds {len(codes_out_df)} rows across "
      f"{codes_out_df['model'].nunique()} model(s).")

Model claude-sonnet-5: 0 lines already done, 342 to process.
  processed 10/342 new lines (saved)
  processed 20/342 new lines (saved)
  processed 30/342 new lines (saved)
  processed 40/342 new lines (saved)
  processed 50/342 new lines (saved)
  processed 60/342 new lines (saved)
  processed 70/342 new lines (saved)
  processed 80/342 new lines (saved)
  processed 90/342 new lines (saved)
  processed 100/342 new lines (saved)
  processed 110/342 new lines (saved)
  processed 120/342 new lines (saved)
  processed 130/342 new lines (saved)
  processed 140/342 new lines (saved)
  processed 150/342 new lines (saved)
  processed 160/342 new lines (saved)
  processed 170/342 new lines (saved)
  processed 180/342 new lines (saved)
  processed 190/342 new lines (saved)
  processed 200/342 new lines (saved)
  processed 210/342 new lines (saved)
  processed 220/342 new lines (saved)
  processed 230/342 new lines (saved)
  processed 240/342 new lines (saved)
  processed 250/342 new lines (saved

## Write the output CSV and summarize

In [14]:
# (Re)price every coded row from the MRF index. Pricing is deterministic and free
# (no API), so we always refresh it -- this keeps prices in sync with the pricing
# logic even for rows coded on an earlier run, and decouples prices from the LLM.
PRICE_COLS = ["price_low", "price_high", "cash_low", "cash_high",
              "n_hospitals", "hospitals", "prices_by_hospital"]
for col in PRICE_COLS:
    if col not in codes_out_df.columns:
        codes_out_df[col] = pd.NA
coded_mask = codes_out_df["code"].notna()
if coded_mask.any():
    repriced = [price_summary(c, t) or {} for c, t in
                zip(codes_out_df.loc[coded_mask, "code"], codes_out_df.loc[coded_mask, "code_type"])]
    for col in PRICE_COLS:
        codes_out_df.loc[coded_mask, col] = [r.get(col) for r in repriced]
    codes_out_df.reindex(columns=COLUMNS).to_csv(OUT_CSV, index=False)
    print(f"(Re)priced {int(coded_mask.sum())} coded rows from the MRF index.")

# The CSV was written incrementally during the run above; here we just summarize.
print(f"Output CSV: {OUT_CSV}  ({len(codes_out_df)} rows)\n")

# Per-model coverage across everything accumulated in the CSV so far.
per_model = codes_out_df.groupby("model").apply(
    lambda g: pd.Series({
        "output_rows": len(g),
        "lines": g[["patient_number", "line_number"]].drop_duplicates().shape[0],
        "lines_with_code": g[g["code"].notna()][["patient_number", "line_number"]]
                            .drop_duplicates().shape[0],
        "distinct_codes": g["code"].nunique(),
    }), include_groups=False
)
display(per_model)

# Detail for the model just run.
this = codes_out_df[codes_out_df["model"] == agent.MODEL]
coded = this[this["code"].notna()]
print(f"\nModel {agent.MODEL}:")
display(coded["code_type"].value_counts().rename("rows").to_frame())
display(coded["confidence"].value_counts(dropna=False).rename("rows").to_frame())

(Re)priced 358 coded rows from the MRF index.
Output CSV: /Users/robbertstruyven/Dropbox/Mac/Downloads/abridge_hackathon/abridge-hackathon/data_processing/processed_csv/avs_line_to_billable_codes.csv  (978 rows)



/var/folders/3s/2l90gqtd4y3656zj00gq24sh0000gn/T/ipykernel_37932/1310046183.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[11.81, 7.65, 9.31, 6.47, 7.77, 9.31, 9.21, 12.47, 3.58, 4.29, 3.77, 4.28, 6.47, 33.38, 938.88, 9.31, 2.64, 6.47, 7.77, 9.11, 10.33, 21.24, 12.58, 14.27, 15.49, 3.88, 4.4, 12.69, 14.39, 11.36, 12.88, 10.42, 11.82, 1.91, 1.91, 8.25, 54.31, 7.12, 30.93, 8.07, 8.09, 9.1, 8.32, 29.86, 17.68, 9.1, 8.32, 29.86, 17.68, 12.38, 8.56, 9.71, 8.56, 9.71, 12.38, 0.49, 70.38, 62.33, 0.49, 96.67, 8.56, 9.71, 8.56, 9.71, 8.56, 9.71, 8.56, 9.71, 6.18, 8.56, 9.71, 8.56, 9.71, 8.56, 9.71, 8.56, 9.71, 11.81, 2.64, 2.99, 2.64, 2.99, 3.38, 7.75, 9.11, 10.33, 12.58, 14.27, 15.49, 21.24, 17.78, 24.06, 21.9, 3.88, 4.4, 17.28, 12.69, 14.39, 11.36, 12.88, 13.42, 9.1, 8.32, 29.86, 17.68, 3.24, 3.67, 3.57, 8.25, 16.91, 14.53, 54.31, 15.08, 7.77, 6.47, 33.5, 7.46, 16.94, 6.18, 11.81, 246.87, 66.52, 2.64, 2.

,output_rows,lines,lines_with_code,distinct_codes
model,,,,
claude-haiku-4-5-20251001,514,342,42,73
claude-sonnet-5,464,342,53,70



Model claude-sonnet-5:


,rows
code_type,
CPT,76
HCPCS,73


,rows
confidence,
high,77
medium,62
low,10


## Inspect one patient: line -> code -> price

Change `PATIENT_NUMBER` to read one encounter's AVS lines with the codes and
cross-hospital price ranges the agent produced.

In [15]:
import json

PATIENT_NUMBER = 0  # choose 0 through 24

one = codes_out_df[
    (codes_out_df["patient_number"] == PATIENT_NUMBER)
    & (codes_out_df["model"] == agent.MODEL)
].copy()
display(Markdown(f"### {one['patient'].iloc[0]} — {one['encounter_date'].iloc[0]}"
                 f"  ·  model: {agent.MODEL}"))

display(Markdown("#### Line -> code (negotiated & self-pay ranges across hospitals)"))
display(
    one[[
        "section", "line_text", "mention", "status",
        "code_type", "code", "code_description", "confidence",
        "price_low", "price_high", "cash_low", "cash_high", "n_hospitals",
    ]].reset_index(drop=True)
)

# Per-hospital detail, exploded from the saved `prices_by_hospital` column.
detail = []
for r in one[one["code"].notna()].itertuples(index=False):
    blob = r.prices_by_hospital
    per_hospital = json.loads(blob) if isinstance(blob, str) and blob else {}
    for label, p in per_hospital.items():
        detail.append({
            "code_type": r.code_type, "code": r.code, "mention": r.mention,
            "hospital_name": p.get("hospital_name", label), "mrf_label": label,
            "cash": p.get("discounted_cash"),
            "neg_mean": p.get("negotiated_mean"), "neg_median": p.get("negotiated_median"),
            "neg_min": p.get("negotiated_min"), "neg_max": p.get("negotiated_max"),
            "n_payers": p.get("n_payers"), "gross": p.get("gross"),
            "min_pub": p.get("min_published"), "max_pub": p.get("max_published"),
            "p50": p.get("p50"), "p10": p.get("p10"), "p90": p.get("p90"),
        })

display(Markdown("#### Price per hospital (per code) — all CMS fields"))
if detail:
    cols = ["code_type", "code", "hospital_name", "mrf_label", "cash",
            "neg_mean", "neg_median", "neg_min", "neg_max", "n_payers",
            "gross", "min_pub", "max_pub", "p50", "p10", "p90"]
    display(pd.DataFrame(detail)[cols].sort_values(["code", "neg_median"]).reset_index(drop=True))
else:
    print("No priceable codes for this patient/model.")

### Ali Jerry Kuhic — 2022-08-05  ·  model: claude-sonnet-5

#### Line -> code (negotiated & self-pay ranges across hospitals)

,section,line_text,mention,status,code_type,code,code_description,confidence,price_low,price_high,cash_low,cash_high,n_hospitals
0,what_we_discussed,Gingivitis,NaN,no_billable_service,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,what_we_discussed,Chronic intractable migraine without aura,NaN,no_billable_service,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,what_we_discussed,Social isolation and stress,NaN,no_billable_service,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,what_we_discussed,Transportation insecurity,NaN,no_billable_service,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,what_we_discussed,Preventive health and screening,NaN,no_billable_service,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,next_steps,Patient referral for dental care placed (sliding-scale clinic).,NaN,no_billable_service,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,next_steps,Daily flossing and twice-daily brushing reviewed.,NaN,no_billable_service,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,next_steps,Medication reconciliation completed and chart list updated.,NaN,no_billable_service,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,next_steps,Keep a simple headache log; return precautions reviewed for any change in pattern or new neurologic symptoms.,NaN,no_billable_service,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,next_steps,Assessment of health and social care needs completed (PRAPARE).,NaN,no_billable_service,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Price per hospital (per code) — all CMS fields

,code_type,code,hospital_name,mrf_label,cash,neg_mean,neg_median,neg_min,neg_max,n_payers,gross,min_pub,max_pub,p50,p10,p90
0,CPT,80061,Cambridge Health Alliance,PriceTransparency20262320,86.0,58.38,86.0,11.81,89.86,154,86.0,11.81,89.86,30.03,0.05,132.9
1,CPT,80061,Boston Children's Longwood,04-2774441_Boston-Childrens-Longwood,233.0,168.69,173.5,52.99,221.35,42,233.0,47.43,221.35,153.17,NaN,NaN


## Compare models

Reads the full output CSV (every model that has been run) and compares how many
lines each model managed to code, how many distinct codes it found, and the
cross-hospital price coverage. Run more tiers (change `MODEL_CHOICE` and rerun)
to fill this table out.

In [16]:
all_df = pd.read_csv(OUT_CSV, dtype={"code": "string", "code_type": "string"})
models = sorted(all_df["model"].dropna().unique())
print(f"{OUT_CSV.name}: {len(all_df)} rows, models = {models}\n")


def per_model_stats(g):
    lines = g[["patient_number", "line_number"]].drop_duplicates()
    coded = g[g["code"].notna()]
    coded_lines = coded[["patient_number", "line_number"]].drop_duplicates()
    return pd.Series({
        "lines_total": len(lines),
        "lines_with_code": len(coded_lines),
        "coverage_%": round(100 * len(coded_lines) / max(len(lines), 1), 1),
        "code_rows": len(coded),
        "distinct_codes": coded["code"].nunique(),
        "avg_hospitals_per_code": round(coded["n_hospitals"].astype("float").mean(), 2)
                                  if len(coded) else None,
    })


compare = all_df.groupby("model").apply(per_model_stats, include_groups=False)
display(Markdown("### Per-model coverage"))
display(compare)

if len(models) > 1:
    # How many models coded each line (agreement / overlap).
    coded = all_df[all_df["code"].notna()]
    per_line = coded.groupby(["patient_number", "line_number"])["model"].nunique()
    display(Markdown("### Line agreement — how many models coded the same line"))
    display(per_line.value_counts().sort_index()
            .rename_axis("n_models_coding_line").rename("n_lines").to_frame())

    # Distinct codes: shared vs unique to each model.
    codes_by_model = {m: set(all_df[(all_df["model"] == m) & all_df["code"].notna()]["code"])
                      for m in models}
    shared = set.intersection(*codes_by_model.values()) if codes_by_model else set()
    display(Markdown("### Distinct codes: shared vs model-specific"))
    display(pd.DataFrame([{
        "model": m,
        "distinct_codes": len(s),
        "shared_by_all": len(s & shared),
        "unique_to_model": len(s - set.union(*(codes_by_model[o] for o in models if o != m))),
    } for m, s in codes_by_model.items()]).set_index("model"))
else:
    print("Only one model present -- run another tier (MODEL_CHOICE) to compare.")

avs_line_to_billable_codes.csv: 978 rows, models = ['claude-haiku-4-5-20251001', 'claude-sonnet-5']



### Per-model coverage

,lines_total,lines_with_code,coverage_%,code_rows,distinct_codes,avg_hospitals_per_code
model,,,,,,
claude-haiku-4-5-20251001,342.0,42.0,12.3,209.0,73.0,1.87
claude-sonnet-5,342.0,53.0,15.5,149.0,70.0,1.88


### Line agreement — how many models coded the same line

,n_lines
n_models_coding_line,
1,27
2,34


### Distinct codes: shared vs model-specific

,distinct_codes,shared_by_all,unique_to_model
model,,,
claude-haiku-4-5-20251001,73,39,34
claude-sonnet-5,70,39,31
